# Export NoobAI XL 1.1 for the browser

This notebook builds the separate ONNX model pack consumed by the GitHub Pages experiment. **It does not run inference in Colab**; it converts the model once and uploads the browser-compatible files to Hugging Face. Actual generation stays in your browser.

## Before running

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU with 16 GB VRAM or more). A high-RAM host runtime is preferable; if the export runs out of system RAM, switch to Colab High-RAM or a machine with at least 24 GB host RAM.
2. Have a Hugging Face account and a **write** access token.
3. Create a public Hugging Face model repository name for the converted pack, for example `YOUR_ACCOUNT/noobai-xl-1.1-browser-onnx`.
4. Review the source model's [license and additional terms](https://huggingface.co/Laxhar/noobai-XL-1.1). The exported pack is a derivative and must retain the applicable model terms.

The exporter loads one component at a time and traces the U-Net at 768×768 to reduce peak activation memory. It still needs a CUDA GPU, several GB of temporary disk space, and enough host memory. If Colab resets or runs out of memory, use a higher-memory GPU/host runtime.


In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/imagegen')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/DragonFireGames/imagegen.git',
        str(repo_dir),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)

%cd /content/imagegen
%pip install -q -r noobai-browser-experiment/tools/requirements-export.txt

import numpy as np
import ml_dtypes
import onnx
print('NumPy:', np.__version__)
print('ml_dtypes:', ml_dtypes.__version__)
print('ONNX:', onnx.__version__)
if not hasattr(ml_dtypes, 'float4_e2m1fn'):
    raise RuntimeError('ml_dtypes is too old for this ONNX version; rerun the dependency cell.')

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f'GPU memory: {free_bytes / 1024**3:.1f} GiB free / {total_bytes / 1024**3:.1f} GiB total')
else:
    raise RuntimeError('No CUDA GPU found. Change the Colab runtime type to a GPU runtime, then rerun this cell.')


In [ ]:
import getpass
import os
import subprocess
import sys

target_repo = input('Public Hugging Face destination repo (OWNER/NAME): ').strip()
if target_repo.count('/') != 1 or not all(target_repo.split('/')):
    raise ValueError('Enter a repo name in OWNER/NAME form.')

token = getpass.getpass('Hugging Face write token (input is hidden): ').strip()
if not token:
    raise ValueError('A Hugging Face write token is required to upload the browser pack.')
os.environ['HF_TOKEN'] = token

command = [
    sys.executable,
    'noobai-browser-experiment/tools/export_noobai_onnx.py',
    '--device', 'cuda',
    '--output', '/content/noobai-xl-1.1-browser-pack',
    '--upload-repo', target_repo,
    '--path-in-repo', 'browser-pack',
]
subprocess.run(command, check=True)

model_pack_url = f'https://huggingface.co/{target_repo}/resolve/main/browser-pack'
print('\nModel-pack URL to paste into the browser experiment:')
print(model_pack_url)


## Next

Open the [NoobAI browser experiment](https://dragonfiregames.github.io/imagegen/noobai-browser-experiment/), paste the model-pack URL printed above, and select **Load and cache model**. The browser downloads the exported ONNX files into OPFS. Subsequent generations use that local cache.

This conversion notebook is only for producing the pack. It does not upload prompts or generated images from the browser.